<a href="https://colab.research.google.com/github/MoAppOfficial/moapp-colab/blob/main/%D8%AA%D8%AD%D9%85%D9%8A%D9%84_%D8%AA%D9%88%D8%B1%D9%86%D8%AA_(%D8%AF%D8%B1%D8%A7%D9%8A%D9%81).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 لوحة تحميل التورنت التفاعلية (جوجل درايف)
#@markdown ---
#@markdown ### 🛠️ 1. إعدادات التثبيت
#@markdown حدد هذا المربع إذا كانت هذه أول مرة تشغل فيها الكود في هذه الجلسة لتثبيت الحزم المطلوبة:
تثبيت_الحزم = True #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🔗 2. طريقة التحميل
طريقة_التحميل = "\u0631\u0627\u0628\u0637 \u0645\u0627\u063A\u0646\u064A\u062A" #@param ["رابط ماغنيت", "رفع ملف تورنت"]
رابط_الماغنيت = "magnet:?xt=urn:btih:56D34D13B5694F6F4530D21C5390A849F2C59640&dn=Foo%20Fighters%20-%20Your%20Favorite%20Toy%20%282026%20Rock%29%20%5BFlac%2024-96%5D%20%5B%20UIndex.org%20%5D&tr=udp://tracker.bittor.pw:1337/announce&tr=udp://tracker.opentrackr.org:1337/announce&tr=udp://tracker.dler.org:6969/announce&tr=udp://open.stealth.si:80/announce&tr=udp://tracker.torrent.eu.org:451/announce&tr=udp://exodus.desync.com:6969/announce&tr=udp://open.demonii.com:1337/announce" #@param {type:"string", placeholder:"ادخل الرابط هنا"}

import os
import sys
import time
from pathlib import Path
from IPython.display import clear_output

# 1. تثبيت الحزم إذا تم تحديد الخيار
if تثبيت_الحزم:
    print("⏳ جاري تثبيت الحزم المطلوبة... يرجى الانتظار.")
    get_ipython().system('python -m pip install -q -U pip setuptools wheel')
    get_ipython().system('python -m pip install -q libtorrent || python -m pip install -q lbry-libtorrent')
    clear_output()
    print("✅ تم تثبيت الحزم بنجاح!\n")

from google.colab import drive, files
import libtorrent as lt

# ---------------------------------------------------------
# الدوال المساعدة
# ---------------------------------------------------------
def human_bytes(value):
    if value is None or value < 0:
        return "?"
    value = float(value)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if value < 1024 or unit == units[-1]:
            if unit == "B":
                return f"{int(value)} {unit}"
            return f"{value:.2f} {unit}"
        value /= 1024.0

def get_drive_root():
    candidates = ["/content/drive/MyDrive", "/content/drive/My Drive"]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise RuntimeError("❌ لم يتم العثور على Google Drive.")

def make_dirs():
    drive.mount("/content/drive", force_remount=True)
    local_root = "/content/torrent"
    drive_root = os.path.join(get_drive_root(), "torrent")
    os.makedirs(local_root, exist_ok=True)
    os.makedirs(drive_root, exist_ok=True)
    job_dir = os.path.join(local_root, f"job_{int(time.time())}")
    os.makedirs(job_dir, exist_ok=True)
    return local_root, drive_root, job_dir

def build_session():
    ses = lt.session()
    try: ses.listen_on(6881, 6891)
    except Exception: pass
    try: ses.start_dht()
    except Exception: pass
    routers = [
        ("router.bittorrent.com", 6881), ("router.utorrent.com", 6881),
        ("dht.transmissionbt.com", 6881), ("router.openbittorrent.com", 6881)
    ]
    for host, port in routers:
        try: ses.add_dht_router(host, port)
        except Exception: pass
    return ses

def add_torrent_from_file(ses, torrent_file_path, save_path):
    ti = lt.torrent_info(torrent_file_path)
    try:
        return ses.add_torrent({"ti": ti, "save_path": save_path})
    except Exception:
        atp = lt.add_torrent_params()
        atp.ti = ti
        atp.save_path = save_path
        return ses.add_torrent(atp)

def add_torrent_from_magnet(ses, magnet_link, save_path):
    try:
        atp = lt.parse_magnet_uri(magnet_link)
        atp.save_path = save_path
        return ses.add_torrent(atp)
    except Exception:
        return lt.add_magnet_uri(ses, magnet_link, {"save_path": save_path})

def choose_torrent_file():
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded.")
    name = next(iter(uploaded.keys()))
    path_candidates = [os.path.join("/content", name), name]
    for candidate in path_candidates:
        if os.path.exists(candidate):
            return candidate
    raise FileNotFoundError("Uploaded file was not found.")

def status_name(state_value):
    names = ["queued", "checking", "metadata", "downloading", "finished", "seeding", "allocating", "checking_resume"]
    try:
        idx = int(state_value)
        if 0 <= idx < len(names): return names[idx]
    except Exception: pass
    return str(state_value)

def one_line_progress(prefix, progress, done, total, speed, seeds, peers, state):
    percent_text = f"{progress:6.2f}%"
    line = (
        f"\r{prefix} | {state:<12} | {percent_text} | "
        f"{human_bytes(done):>10} / {human_bytes(total):<10} | "
        f"{human_bytes(speed):>10}/s | seeds {str(seeds):<5} | peers {str(peers):<5}"
    )
    sys.stdout.write(line[:220].ljust(220))
    sys.stdout.flush()

def select_files(handle):
    print("\n⏳ جاري إحضار بيانات الملفات (البيانات الوصفية)... يرجى الانتظار.")
    while not handle.status().has_metadata:
        time.sleep(1)

    info = handle.get_torrent_info()
    num_files = info.num_files()

    print("\n" + "="*50)
    print("📁 قائمة الملفات المتاحة للتحميل:")
    print("="*50)
    for i in range(num_files):
        file_path = info.files().file_path(i)
        file_size = human_bytes(info.files().file_size(i))
        print(f"[{i}] {file_path} ({file_size})")
    print("="*50)

    choice = input("\n✍️ أدخل أرقام الملفات التي تريدها مفصولة بفاصلة (مثال: 0,2). أو اتركه فارغاً لتحميل الكل: ").strip()
    selected_paths = []

    if choice:
        try:
            selected_indices = [int(x.strip()) for x in choice.split(',') if x.strip().isdigit()]
            priorities = [0] * num_files
            for idx in selected_indices:
                if 0 <= idx < num_files:
                    priorities[idx] = 1
                    selected_paths.append(info.files().file_path(idx))
            handle.prioritize_files(priorities)
            print(f"\n✅ تم اختيار {len(selected_indices)} ملف/ات للتحميل.\n")
        except Exception as e:
            print(f"\n⚠️ خطأ في الإدخال، سيتم تحميل جميع الملفات.\n")
            for i in range(num_files): selected_paths.append(info.files().file_path(i))
    else:
        print("\n✅ سيتم تحميل جميع الملفات.\n")
        for i in range(num_files): selected_paths.append(info.files().file_path(i))

    return selected_paths

def wait_until_finished(handle):
    finished_states = set()
    try: finished_states.add(int(lt.torrent_status.seeding))
    except Exception: pass
    try: finished_states.add(int(lt.torrent_status.finished))
    except Exception: pass

    while True:
        s = handle.status()
        progress = float(getattr(s, "progress", 0.0)) * 100.0
        total = getattr(s, "total_wanted", 0) or getattr(s, "total", 0) or 0
        done = getattr(s, "total_wanted_done", 0) or getattr(s, "total_done", 0) or 0
        speed = getattr(s, "download_rate", 0) or 0
        peers = getattr(s, "num_peers", 0) or 0
        seeds = getattr(s, "num_seeds", 0) or getattr(s, "list_seeds", 0) or 0
        state_value = getattr(s, "state", 0)
        state_text = status_name(state_value)

        one_line_progress("Download", progress, done, total, speed, seeds, peers, state_text)

        if progress >= 100.0 and state_text in {"finished", "seeding"}: break
        try:
            if int(state_value) in finished_states: break
        except Exception: pass

        error_obj = getattr(s, "errc", None)
        if error_obj:
            try:
                if error_obj.value() != 0: raise RuntimeError(error_obj.message())
            except Exception: pass
        time.sleep(1)
    sys.stdout.write("\n")
    sys.stdout.flush()

def copy_file_with_progress(src_file, dst_file, copied_so_far=0, grand_total=None):
    os.makedirs(os.path.dirname(dst_file), exist_ok=True)
    chunk_size = 4 * 1024 * 1024
    file_size = os.path.getsize(src_file)
    copied_local = 0
    last_time = time.time()
    last_bytes = copied_so_far

    with open(src_file, "rb") as fsrc, open(dst_file, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk: break
            fdst.write(chunk)
            copied_local += len(chunk)
            copied_global = copied_so_far + copied_local

            now = time.time()
            if now - last_time >= 0.2:
                delta = copied_global - last_bytes
                speed = delta / (now - last_time) if now > last_time else 0
                progress = (copied_global / grand_total * 100.0) if grand_total else 0.0
                one_line_progress("Copy", progress, copied_global, grand_total or file_size, speed, "-", "-", "copying")
                last_time = now
                last_bytes = copied_global

    one_line_progress("Copy", 100.0, grand_total or file_size, grand_total or file_size, 0, "-", "-", "done")
    return copied_so_far + copied_local

# ---------------------------------------------------------
# التشغيل الرئيسي
# ---------------------------------------------------------
def main():
    local_root, drive_root, job_dir = make_dirs()
    ses = build_session()

    if طريقة_التحميل == "رفع ملف تورنت":
        print("📤 يرجى رفع ملف التورنت الخاص بك الآن...")
        torrent_file_path = choose_torrent_file()
        handle = add_torrent_from_file(ses, torrent_file_path, job_dir)
    elif طريقة_التحميل == "رابط ماغنيت":
        if not رابط_الماغنيت or not رابط_الماغنيت.startswith("magnet:?"):
            print("❌ خطأ: يرجى وضع رابط ماغنيت صحيح في اللوحة بالأعلى.")
            return
        print("🧲 جاري قراءة رابط الـ الماغنيت...")
        handle = add_torrent_from_magnet(ses, رابط_الماغنيت, job_dir)

    # اختيار الملفات
    selected_paths = select_files(handle)

    # التحميل
    wait_until_finished(handle)
    try: handle.pause()
    except Exception: pass
    time.sleep(2)

    # النسخ إلى درايف
    print("\n🚀 جاري نسخ الملفات المحددة فقط إلى جوجل درايف...")
    copied_paths = []
    for rel_path in selected_paths:
        src_file = os.path.join(job_dir, rel_path)
        dst_file = os.path.join(drive_root, rel_path)
        if os.path.isfile(src_file):
            total_size = os.path.getsize(src_file)
            copy_file_with_progress(src_file, dst_file, 0, total_size)
            sys.stdout.write("\n")
            copied_paths.append(dst_file)

    print("\n🎉 انتهت العملية بنجاح!")
    print(f"📂 مسار الملفات في درايف: {drive_root}")

main()

✅ تم تثبيت الحزم بنجاح!

Mounted at /content/drive
🧲 جاري قراءة رابط الـ الماغنيت...

⏳ جاري إحضار بيانات الملفات (البيانات الوصفية)... يرجى الانتظار.


/tmp/ipykernel_22215/2253875490.py:63: DeprecationWarning: listen_on() is deprecated
  try: ses.listen_on(6881, 6891)
/tmp/ipykernel_22215/2253875490.py:65: DeprecationWarning: start_dht() is deprecated
  try: ses.start_dht()
/tmp/ipykernel_22215/2253875490.py:72: DeprecationWarning: add_dht_router() is deprecated
  try: ses.add_dht_router(host, port)
/tmp/ipykernel_22215/2253875490.py:128: DeprecationWarning: get_torrent_info() is deprecated
  info = handle.get_torrent_info()



📁 قائمة الملفات المتاحة للتحميل:
[0] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/00.Info.m3u (405 B)
[1] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/00.Info.nfo (1.67 KB)
[2] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/01. Foo Fighters - Caught In The Echo.flac (82.59 MB)
[3] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/02. Foo Fighters - Of All People.flac (55.08 MB)
[4] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/03. Foo Fighters - Window.flac (67.84 MB)
[5] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/04. Foo Fighters - Your Favorite Toy.flac (66.49 MB)
[6] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/05. Foo Fighters - If You Only Knew.flac (82.53 MB)
[7] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/06. Foo Fighters - Spit Shine.flac (72.51 MB)
[8] Foo Fighters - Your Favorite Toy (2026 Rock) [Flac 24-96]/07. Foo Fighters - Unconditional.flac (82.73 MB)
[9] Foo Fighters - Yo